# **Exploratory Data Analysis**

## **City Car-Sharing Rental Dataset**

### **Objective**

The goal of this analysis is to:

1. Understand **fleet utilization and rental demand patterns**.
2. Analyze **battery behavior and customer preferences**.
3. Identify **charging inefficiencies and operational interventions**.
4. Support decisions about **when to move vehicles from charging stations** to avoid additional fees.

---

### 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

### 2. Load Dataset


In [ ]:
data = pd.read_csv("Customer_Vehicles.csv")

data.head()

### 3. Data Preparation


#### Convert Timestamp Columns


In [ ]:
data['STARTED_TIME'] = pd.to_datetime(data['STARTED_TIME'])
data['FINISHED_TIME'] = pd.to_datetime(data['FINISHED_TIME'])

#### Create Derived Features

We create additional variables that will help in analyzing rental patterns.

In [ ]:
# Rental duration in minutes
data['duration(mts)'] = (data['FINISHED_TIME'] - data['STARTED_TIME']).dt.total_seconds() / 60

# Extract month and day of week
data['month'] = data['STARTED_TIME'].dt.month
data['day'] = data['STARTED_TIME'].dt.day_name()

data.head()

#### Check Missing Values


In [ ]:
data.isna().sum()

Fill missing values if required.

In [ ]:
data.fillna(0, inplace=True)

#### Unique Values in Dataset

In [ ]:
data.nunique()

### 4. Fleet Utilization Analysis
This section analyzes **rental demand patterns** to identify peak usage periods.


#### 4.1 Total Rentals


In [ ]:
total_rentals = len(data)
print("Total Rentals:", total_rentals)

#### 4.2 Rentals by Day (Customer vs Service)

Service rentals represent **operational activities**, while customer rentals represent **actual demand**.

In [ ]:
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

daily = data.groupby(['day','SERVICERENTAL']).size().unstack().reindex(order)

daily.plot(kind='bar')

plt.title("Rentals by Day (Customer vs Service)")
plt.xlabel("Day")
plt.ylabel("Number of Rentals")

plt.show()

Insight:

* Helps identify **high-demand days**.
* Separates **customer demand vs operational activity**.

#### 4.3 Monthly Rental Trends


In [ ]:
monthly = data.groupby(['month','SERVICERENTAL']).size().unstack()

monthly.plot(kind='line')

plt.title("Monthly Rentals: Customer vs Service")
plt.xlabel("Month")
plt.ylabel("Number of Rentals")
plt.legend(['Customer Rentals','Service Rentals'])

plt.show()

Insight:

* Identifies **seasonal rental patterns**.

### 5. Battery Behavior Analysis

This section examines **customer preferences regarding battery levels**.

#### 5.1 Battery Level Categories


In [ ]:
bins = [0,20,40,60,80,100]
labels = ['0-20','20-40','40-60','60-80','80-100']

data['battery_range'] = pd.cut(data['CHARGELEVELSTART'], bins=bins, labels=labels)

#### 5.2 Customer Rentals by Battery Level

Only customer rentals are considered to avoid operational bias.

In [ ]:
customer = data[data['SERVICERENTAL'] == False]

battery_counts = customer.groupby('battery_range').size()

sns.barplot(x=battery_counts.index, y=battery_counts.values)

plt.title("Customer Rentals by Battery Level")
plt.xlabel("Battery Level")
plt.ylabel("Number of Rentals")

plt.show()

Insight:

* Reveals **battery levels customers prefer when renting vehicles**.

### 6. Charging Behavior Analysis

This section evaluates **charging efficiency and interruptions**.

#### 6.1 Charging Interruptions

Charging interruptions occur when a vehicle is **plugged into a charger but does not reach 100% battery**, often because a customer rents the vehicle.


In [ ]:
charging = data[data['CHARGED'] == True]

interruptions = charging[charging['CHARGELEVELEND'] < 100]

pie_data = [len(interruptions), len(charging) - len(interruptions)]
labels = ['Interrupted Charging','Completed Charging']
plt.pie(pie_data, labels=labels, autopct='%1.1f%%')
plt.title("Charging Interruptions")

plt.show()

Insight:

* Indicates **how often vehicles are rented while charging**


#### 6.2 Charging Events

In [ ]:
charging_counts = data['CHARGED'].value_counts()

sns.barplot(x=charging_counts.index.astype(str), y=charging_counts.values)

plt.title("Charging Events")
plt.xlabel("Vehicle Plugged into Charger")
plt.ylabel("Number of Events")

plt.show()

Insight:

* Shows how frequently vehicles **use charging stations**.

### 7. Charging Efficiency

#### 7.1 Vehicles Reaching Full Charge


In [ ]:
full_charge = data[data['CHARGELEVELEND'] == 100]

vehicle_counts = full_charge['VEHICLE_ID'].value_counts()

sns.barplot(x=vehicle_counts.values, y=vehicle_counts.index)

plt.title("Vehicles Reaching 100% Charge")
plt.xlabel("Number of Full Charges")
plt.ylabel("Vehicle ID")

plt.show()

Insight:

* Identifies vehicles that **occupy charging stations most frequently**.

#### 7.2 Service Agent Interventions

Service rentals indicate **operational actions such as vehicle relocation or charging**.

In [ ]:
service_counts = data['SERVICERENTAL'].value_counts()

sns.barplot(x=service_counts.index.astype(str), y=service_counts.values)

plt.title("Service Agent Interventions")
plt.xlabel("Service Rental")
plt.ylabel("Number of Trips")

plt.show()

Insight:

* Shows **how often service agents interact with vehicles**.

## **Key Observations**

Based on the analysis:

* Rental demand varies across **days and months**.
* Customers tend to rent vehicles with **moderate battery levels**.
* Some vehicles are rented **before reaching full charge**, causing charging interruptions.
* Service agents frequently intervene to **manage fleet operations**.

These insights help determine **optimal charging management strategies**.